# Severity models: pressure-injury stage, burn depth, diabetic-foot Wagner grade

Trains the three severity heads the app shows next to the wound type, each with 5-fold cross-validation and scored on a
locked test set, starting from the outline run's wound-type model (which already learnt wounds from ~4,800 photos).

**Data: every public photo with these labels** (~16,000 raw photos from 10 Kaggle datasets). Several datasets are mostly
augmented copies of each other, so copies are found by image content (rotation/flip-invariant embeddings) and kept in one
fold; test photos come only from datasets that look original, and any training photo resembling a test photo is
removed. The log shows raw photos, unique photos and how alike they are, so the scores aren't inflated by copies.

A head is installed only if its mean test macro-F1 reaches 0.6; otherwise the app keeps saying nothing about it.

**Before you start:** GPU T4 x2, Internet on. Datasets and your outline run are attached through the notebook's settings.
Research use only: most of these datasets state no licence.

## 1. Code (branch `wound-ai-severity`)

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 --branch wound-ai-severity https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Datasets

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/link_kaggle_inputs.py                  # public sets (pressure-ulcer stages, YOLO burn boxes)
python scripts/link_kaggle_inputs.py --set severity   # stage / depth / grade sets

## 3. The wound-type model (starting point)

In [ ]:
import glob, shutil, os
os.chdir("/kaggle/working/wound_1/wound-ai")
os.makedirs("checkpoints", exist_ok=True)
# the real checkpoint is ~110 MB; take the largest copy
found = sorted(glob.glob("/kaggle/input/**/best/wound_type.pt", recursive=True), key=os.path.getsize, reverse=True)
if not found:
    raise SystemExit("wound_type.pt not found: attach your outline run (notebook2f5893b152) as an input.")
shutil.copy(found[0], "checkpoints/wound_type.pt")
print("using", found[0], os.path.getsize(found[0]) // 2**20, "MB")

## 4. Build the severity data (copies grouped, test locked; ~5 min)

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/build_severity_dataset.py --out data 2>&1 | tee /kaggle/working/build_log.txt

## 5. Smoke test (a few minutes)

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/severity_cv.py --manifest data/severity_manifest.csv --init checkpoints/wound_type.pt \
  --folds 2 --smoke --out /kaggle/working/severity_smoke && rm -rf /kaggle/working/severity_smoke

## 6. Cross-validation (~2–4 h)
Re-running the same cell continues where it stopped.

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/severity_cv.py --manifest data/severity_manifest.csv --init checkpoints/wound_type.pt \
  --folds 5 --size 384 --batch 32 --epochs 30 --patience 8 --out /kaggle/working/severity_cv

## 7. Package the results

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/outputs
cd /kaggle/working/severity_cv
cp summary.md summary.json /kaggle/working/outputs/
for d in best untrusted; do [ -d $d ] && cp -r $d /kaggle/working/outputs/; done
for h in */fold*; do
  [ -f $h/eval_test.json ] && cp $h/eval_test.json /kaggle/working/outputs/$(echo $h | tr / _)_test.json
done
cp /kaggle/working/build_log.txt /kaggle/working/wound_1/wound-ai/data/severity_manifest.csv /kaggle/working/outputs/
cat /kaggle/working/outputs/summary.md
ls -laR /kaggle/working/outputs | head -40